# PHASE 6: Model Evaluation, Interpretation & Deployment
**Traceability**
- Issue ID: #6 Model Evaluation, Interpretation & Deployment

## 1. Objectives
- Provide a comprehensive evaluation of all trained models on the test set.
- Interpret the model's decision-making process using SHAP (SHapley Additive exPlanations).
- Quantify prediction uncertainty using Quantile Regression to support maintenance decisions.
- Summarize the final pipeline and provide deployment-ready artifacts.

### 6.1 Import Libraries & Load Artifacts
We load the best models and scalers from previous phases to evaluate them on the unseen test data.

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
import shap
import warnings

warnings.filterwarnings('ignore')

# ── Reproducibility Config ──────────────────────────────────────────────
np.random.seed(42)

# ── Global Config ────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../data/processed')
ARTIFACTS_DIR = Path('../artifacts')
FIGURES_DIR = Path('../results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load Data & Artifacts
df_test = pd.read_csv(PROCESSED_DIR / 'test_labeled.csv')
with open(ARTIFACTS_DIR / 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open(ARTIFACTS_DIR / 'best_regressor.pkl', 'rb') as f:
    best_reg = pickle.load(f)

feature_cols = [c for c in df_test.columns if not any(x in c for x in ['unit_number', 'RUL', 'label'])]
X_te = scaler.transform(df_test[feature_cols])
y_te_rul = df_test['RUL'].values

# For "Last Cycle" evaluation
df_test_last = df_test.groupby('unit_number').last().reset_index()
X_te_last = scaler.transform(df_test_last[feature_cols])
y_te_last_rul = df_test_last['RUL'].values

### 6.2 Model Interpretability (SHAP)
Use SHAP values to explain the contribution of each feature to the model's predictions, both globally and for individual engine instances.

In [ ]:
print("\n--- Model Interpretability (SHAP) ---")
explainer = shap.TreeExplainer(best_reg)
shap_values = explainer.shap_values(X_te[:500])

# Global Summary Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_te[:500], feature_names=feature_cols, show=False)
plt.title('SHAP Summary (Global Feature Importance)')
plt.tight_layout()
plt.show()

# Single Engine Waterfall Plot (Near-failure case)
near_failure_idx = np.where(y_te_rul < 30)[0][0]
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap.Explanation(
    values=explainer.shap_values(X_te[near_failure_idx:near_failure_idx+1])[0],
    base_values=explainer.expected_value,
    data=X_te[near_failure_idx],
    feature_names=feature_cols
), show=False)
plt.title(f'SHAP Waterfall (Engine Near Failure, RUL={y_te_rul[near_failure_idx]})')
plt.tight_layout()
plt.show()

### 6.3 Uncertainty Quantification
Train Quantile Regression models to provide an 80% confidence interval for RUL predictions, giving maintenance managers a range of probable outcomes.

In [ ]:
print("\n--- Uncertainty Quantification (Quantile Regression) ---")
df_train = pd.read_csv(PROCESSED_DIR / 'train_labeled.csv')
X_tr = scaler.transform(df_train[feature_cols])
y_tr_rul = df_train['RUL'].values

quantile_models = {}
for q in [0.10, 0.50, 0.90]:
    q_model = lgb.LGBMRegressor(objective='quantile', alpha=q, n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)
    q_model.fit(X_tr, y_tr_rul)
    quantile_models[q] = q_model

preds_q10 = quantile_models[0.10].predict(X_te_last)
preds_q50 = quantile_models[0.50].predict(X_te_last)
preds_q90 = quantile_models[0.90].predict(X_te_last)

# Visualization of Uncertainty Bands
plt.figure(figsize=(14, 6))
n = 50
plt.fill_between(range(n), preds_q10[:n], preds_q90[:n], alpha=0.3, color='#2E75B6', label='80% Confidence Band (Q10–Q90)')
plt.plot(preds_q50[:n], color='#1F4E79', linewidth=1.5, marker='o', label='Median Prediction (Q50)')
plt.plot(y_te_last_rul[:n], color='#FF7043', linewidth=1.5, linestyle='--', marker='x', label='Actual RUL')
plt.title('RUL Prediction with 80% Uncertainty Bands (Test Set — Last Cycles)')
plt.xlabel('Engine Index'); plt.ylabel('RUL (cycles)'); plt.legend()
plt.tight_layout()
plt.show()

### 6.4 Final Evaluation Summary
Generate a summary of the best model's performance on the test set.

In [ ]:
print("\n--- Final Model Comparison Summary ---")
test_preds = best_reg.predict(X_te_last)
rmse = np.sqrt(mean_squared_error(y_te_last_rul, test_preds))
r2 = r2_score(y_te_last_rul, test_preds)

summary_df = pd.DataFrame({
    'Metric': ['Best Test RMSE', 'Best Test R2', 'Total Features', 'Total Engines'],
    'Value': [f"{rmse:.2f}", f"{r2:.3f}", len(feature_cols), 100]
})
print(summary_df.to_string(index=False))